# Chapter 11 &mdash; Inherently Ambiguous Languages, and the Onion/Lasso Idioms

**Concept 11 of the Chapter 11 decomposition:** *Inherently Ambiguous Languages, and the Onion/Lasso Idioms*

$\{a^ib^jc^k : i=j \text{ or } j=k\}$ has no unambiguous grammar at all.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Inherently-Ambiguous/Concept-Inherently-Ambiguous.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Concept 10 fixed an ambiguous *grammar*. Some **languages** cannot be fixed:
$$L_{amb} = \{a^ib^jc^k : i=j \ \text{or}\ j=k\}$$
is **inherently ambiguous** &mdash; *every* CFG for it is ambiguous. The reason is
visible in the definition: strings with $i=j=k$ belong for **two independent
reasons**, and a grammar must offer a derivation for each.

Building the grammar teaches two reusable idioms:

* the **onion** &mdash; `X -> aXb | ''` wraps matching layers around a core;
* the **lasso** &mdash; `Y -> cY | ''` piles on an unconstrained tail.

Each disjunct is an onion joined to a lasso; the union of the two is the grammar, and
the overlap is where the ambiguity lives.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The grammar, built from the two idioms

In [ ]:
# i = j branch:  (a^i b^i) c^k     onion on a/b, lasso on c
# j = k branch:  a^i (b^j c^j)     lasso on a, onion on b/c
Lamb = mkg({'S':  ["XC", "AY"],
            'X':  ["", "aXb"],      # onion: a^i b^i
            'C':  ["", "cC"],       # lasso: c^k
            'A':  ["", "aA"],       # lasso: a^i
            'Y':  ["", "bYc"]})     # onion: b^j c^j
show(Lamb)

### The specification

In [ ]:
def in_Lamb(s):
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    if rest[j:] != 'c' * k: return False
    if s != 'a'*i + 'b'*j + 'c'*k: return False
    return i == j or j == k

## 3. Tests

The grammar is correct.

In [ ]:
from itertools import product
L = set(language(Lamb, 6))
want = {''.join(p) for n in range(7) for p in product('abc', repeat=n)
        if in_Lamb(''.join(p))}
print("generated %d, intended %d" % (len(L), len(want)))
print("missing :", sorted(want - L)[:5], " extra :", sorted(L - want)[:5])
assert L == want

**Strings with $i=j=k$ have two derivations** &mdash; one per reason for belonging.

In [ ]:
for w in ['', 'abc', 'aabbcc', 'aaabbbccc']:
    print("  %-12r parse trees : %d" % (w, nparses(Lamb, w)))
assert nparses(Lamb, 'abc') >= 2
print("\nOne tree uses S -> XC (because i=j); the other S -> AY (because j=k).")

Strings belonging for only **one** reason have only one tree.

In [ ]:
for w in ['aabbc', 'abbcc', 'aabbbccc']:
    i = len(w) - len(w.lstrip('a')); rest = w[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    print("  %-10r i=%d j=%d k=%d  trees %d" % (w, i, j, k, nparses(Lamb, w)))
assert nparses(Lamb, 'aabbc') == 1

**The onion idiom**, alone.

In [ ]:
Onion = mkg({'X': ["", "aXb"]}, 'X')
print("onion language :", language(Onion, 6))
assert all(w == 'a'*(len(w)//2) + 'b'*(len(w)//2) for w in language(Onion, 6))

**The lasso idiom**, alone.

In [ ]:
Lasso = mkg({'C': ["", "cC"]}, 'C')
print("lasso language :", language(Lasso, 5))
assert language(Lasso, 5) == ['', 'c', 'cc', 'ccc', 'cccc', 'ccccc']
print("\nOnion = matched growth.  Lasso = free growth.  Most CFG are made of these.")

Layering cannot help here: the two reasons are genuinely independent.

In [ ]:
print("For 1+2*3 the two trees meant two ANSWERS -- so one had to be wrong.")
print("For a^n b^n c^n the two trees mean two REASONS -- both are right.")
print("No reformulation can merge them, which is what 'inherently' means.")

## 4. Exercises


1. Give the two parse trees of `aabbcc` explicitly.
2. Is $\{a^ib^jc^k : i=j\}$ inherently ambiguous? Why not?
3. Name another inherently ambiguous language.

In [ ]:
# Your work for the exercises above.